# Step 10 — Gradio Demo

Build a web interface that connects to the SageMaker Endpoint.

**Prerequisite:** Endpoint must be running (see notebook 08).  
**Public URL:** `share=True` generates a link valid for 1 week.

In [ ]:
import subprocess
subprocess.run(["pip", "install", "gradio", "-q"])

In [ ]:
import gradio as gr
import pickle
import boto3

BUCKET = "YOUR-S3-BUCKET-NAME"
REGION = "us-east-2"
ENDPOINT_NAME = "symptom-classifier-endpoint"

# Download label encoder
s3 = boto3.client("s3", region_name=REGION)
s3.download_file(BUCKET, "symptom-data/processed/label_encoder.pkl", "label_encoder.pkl")
with open("label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

print(f"Label encoder loaded — {len(le.classes_)} classes ✅")

In [ ]:
import sagemaker
from sagemaker.predictor import Predictor
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

boto_session = boto3.Session(region_name=REGION)
session = sagemaker.Session(boto_session=boto_session, default_bucket=BUCKET)

predictor = Predictor(
    endpoint_name=ENDPOINT_NAME,
    sagemaker_session=session,
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer(),
)

def predict_disease(symptom_text: str) -> dict:
    if not symptom_text.strip():
        return {}
    # top_k=22 returns all classes (not just the top 1)
    response = predictor.predict({
        "inputs": symptom_text,
        "parameters": {"top_k": 22}
    })
    results = {}
    for item in response[:5]:
        label_num = int(item["label"].split("_")[1])
        disease   = le.classes_[label_num]
        results[disease] = round(item["score"], 3)
    return results

demo = gr.Interface(
    fn=predict_disease,
    inputs=gr.Textbox(
        label="Describe your symptoms",
        lines=3,
        placeholder="e.g. I have severe headaches and sensitivity to light..."
    ),
    outputs=gr.Label(num_top_classes=5, label="Possible Conditions"),
    title="Medical Symptom Classifier",
    description="For educational purposes only — not a substitute for medical advice.",
    examples=[
        ["I have severe headaches and sensitivity to light"],
        ["My joints are swollen and painful especially in the morning"],
        ["I feel very thirsty, urinate frequently and feel tired all the time"],
        ["I have a persistent cough, fever and difficulty breathing"],
    ]
)

demo.launch(share=True)  # Generates a public URL valid for 1 week